# 05 — Results and Discussion

Read-only synthesis of notebooks 01–04. This notebook does **not** retrain any models — it consumes the saved prediction CSVs and joblib files and produces the headline tables, plots, and discussion that the project report draws from.

**Pipeline recap:**
1. **01_data_loading** — clean RECS 2020 (18,495 households × 69 cols), derive end-use $ columns, define `efficiency_class` as climate-stratified tertile of `cost_per_sqft`.
2. **02_eda** — weighted distribution / correlation analysis; confirmed size proxies dominate `TOTALDOL` and validated per-sqft normalization.
3. **03_classification** — Multinomial logistic regression on `efficiency_class` (3 classes × 7 climate groups, ~216 OHE features).
4. **04_regression** — Elastic Net + Gradient Boosting on `TOTALDOL`, both with grid-searched hyperparameters and weighted metrics.

Every metric in this notebook uses `sample_weight = NWEIGHT` so the numbers are population-scaled.

## Setup

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, mean_squared_error, r2_score,
                             mean_absolute_error)

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

DATA_DIR   = '../data'
MODELS_DIR = '../models'


In [ ]:
df       = pd.read_pickle(os.path.join(DATA_DIR, 'recs2020_clean.pkl'))
split    = json.load(open(os.path.join(DATA_DIR, 'train_test_split.json')))
clf_pred = pd.read_csv(os.path.join(MODELS_DIR, 'logreg_test_predictions.csv'))
reg_pred = pd.read_csv(os.path.join(MODELS_DIR, 'regression_test_predictions.csv'))

print('df       :', df.shape)
print('clf_pred :', clf_pred.shape)
print('reg_pred :', reg_pred.shape)


## Classification — predicting `efficiency_class`

Three balanced classes within each climate group (efficient / average / inefficient). Random-baseline accuracy ≈ 33.3%. The model uses 55 features, ~216 after one-hot encoding.

In [ ]:
y_true = clf_pred['true']
y_pred = clf_pred['pred']
w      = clf_pred['NWEIGHT']
labels = ['efficient', 'average', 'inefficient']

acc = accuracy_score(y_true, y_pred, sample_weight=w)
prec, rec, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, sample_weight=w, zero_division=0)

overall = pd.DataFrame({
    'class':     labels,
    'precision': prec,
    'recall':    rec,
    'f1':        f1,
    'support':   support,
})
print(f"Weighted overall accuracy: {acc:.3f}")
print(f"(random baseline ≈ 0.333 — three balanced classes)")
print()
print(overall.to_string(index=False))


In [ ]:
# Confusion matrix (weighted) as a heatmap
cm = confusion_matrix(y_true, y_pred, labels=labels, sample_weight=w)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_norm, annot=cm_norm, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            cbar_kws={'label': 'row-normalized share'}, ax=ax)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title(f'Classification confusion matrix (weighted, accuracy = {acc:.2%})')
plt.tight_layout()
plt.show()


In [ ]:
# Per-climate accuracy
per_clim = (clf_pred.groupby('BA_climate_grp', observed=True)
                    .apply(lambda g: accuracy_score(g['true'], g['pred'],
                                                    sample_weight=g['NWEIGHT']))
                    .rename('weighted_accuracy')
                    .to_frame()
                    .join(clf_pred.groupby('BA_climate_grp', observed=True)
                                  .size().rename('n_test')))
per_clim = per_clim.sort_values('weighted_accuracy', ascending=True)
print(per_clim)

fig, ax = plt.subplots(figsize=(8, 4))
colors = sns.color_palette('viridis', n_colors=len(per_clim))
ax.barh(per_clim.index, per_clim['weighted_accuracy'], color=colors)
ax.axvline(1/3, color='red', linestyle='--', linewidth=1, label='random baseline')
ax.set_xlabel('Weighted accuracy')
ax.set_title('Classification accuracy by climate group')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Regression — predicting `TOTALDOL`

We compare two models head-to-head on the same held-out test set:
- **Elastic Net** — combined L1/L2 regularization, linear, fast.
- **Gradient Boosting** — captures nonlinear interactions but more expensive.

All metrics are weighted by `NWEIGHT`.

In [ ]:
def reg_metrics(y, y_pred, w):
    rmse = float(np.sqrt(mean_squared_error(y, y_pred, sample_weight=w)))
    mae  = float(mean_absolute_error(y, y_pred, sample_weight=w))
    r2   = float(r2_score(y, y_pred, sample_weight=w))
    bias = float(np.average(y_pred - y, weights=w))
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'mean_bias': bias}

y     = reg_pred['TOTALDOL_true']
w_reg = reg_pred['NWEIGHT']
metric_table = pd.DataFrame({
    'Elastic Net':       reg_metrics(y, reg_pred['pred_enet'], w_reg),
    'Gradient Boosting': reg_metrics(y, reg_pred['pred_gbr'],  w_reg),
}).T
print(metric_table)


In [ ]:
# Per-climate RMSE comparison
def per_climate_rmse(col):
    return (reg_pred.groupby('BA_climate_grp', observed=True)
            .apply(lambda g: float(np.sqrt(mean_squared_error(
                g['TOTALDOL_true'], g[col], sample_weight=g['NWEIGHT']))))
            .rename(col))

clim_rmse = pd.concat([per_climate_rmse('pred_enet'),
                       per_climate_rmse('pred_gbr')], axis=1)
clim_rmse.columns = ['Elastic Net', 'Gradient Boosting']
clim_rmse = clim_rmse.sort_values('Gradient Boosting', ascending=True)
print(clim_rmse)

fig, ax = plt.subplots(figsize=(9, 4.5))
clim_rmse.plot(kind='barh', ax=ax, color=['#3c8ec0', '#cc4c4c'])
ax.set_xlabel('Weighted RMSE ($)')
ax.set_title('RMSE by climate group — Elastic Net vs. Gradient Boosting')
plt.tight_layout()
plt.show()


In [ ]:
# Diagnostic plot for the better model
better = ('Gradient Boosting' if metric_table.loc['Gradient Boosting', 'RMSE']
          < metric_table.loc['Elastic Net', 'RMSE'] else 'Elastic Net')
pred_col = 'pred_gbr' if better == 'Gradient Boosting' else 'pred_enet'
y_pred = reg_pred[pred_col]
resid  = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(y, y_pred, s=4, alpha=0.25, color='steelblue')
mn, mx = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
axes[0].plot([mn, mx], [mn, mx], 'k--', linewidth=1)
axes[0].set_xlabel('Actual TOTALDOL ($)')
axes[0].set_ylabel('Predicted TOTALDOL ($)')
axes[0].set_title(f'{better}: predicted vs. actual (n={len(y):,})')

axes[1].hist(resid, bins=80, color='steelblue', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Residual: actual − predicted ($)')
axes[1].set_ylabel('Households')
axes[1].set_title(f'{better}: residual distribution (mean bias '
                  f'${metric_table.loc[better, "mean_bias"]:,.0f})')
plt.tight_layout()
plt.show()


## Cross-cutting analysis — do the two layers tell a coherent story?

The classifier sorts households into `efficiency_class` tertiles (defined on cost-per-sqft within climate). The regressor predicts total annual dollars. These two views are **not** identical — a small drafty home can be classified inefficient while still having a low absolute bill. We check the alignment empirically.

In [ ]:
joined = clf_pred.merge(reg_pred[['DOEID','pred_gbr','TOTALDOL_true']], on='DOEID')

ct = (joined.groupby('pred', observed=True)
            .agg(n=('DOEID','count'),
                 mean_pred_dol=('pred_gbr','mean'),
                 mean_true_dol=('TOTALDOL_true','mean'))
            .reindex(['efficient','average','inefficient']))
print(ct)


In [ ]:
# Visualize the relationship: predicted-class boxplot of predicted TOTALDOL
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=joined, x='pred', y='pred_gbr',
            order=['efficient','average','inefficient'],
            hue='pred', palette='RdYlGn_r', legend=False, ax=ax)
ax.set_ylim(0, joined['pred_gbr'].quantile(0.99))
ax.set_xlabel('Predicted efficiency class')
ax.set_ylabel('Predicted TOTALDOL ($)')
ax.set_title('Predicted annual cost distribution within each predicted class')
plt.tight_layout()
plt.show()


In [ ]:
# Walk-through: highest-bill test household
focus_id = joined.sort_values('TOTALDOL_true', ascending=False)['DOEID'].iloc[0]
focus_meta = joined[joined['DOEID'] == focus_id].iloc[0]
print(f"DOEID {focus_id}")
print(f"  climate           : {focus_meta['BA_climate_grp']}")
print(f"  true class        : {focus_meta['true']}")
print(f"  predicted class   : {focus_meta['pred']}")
print(f"  actual TOTALDOL   : ${focus_meta['TOTALDOL_true']:,.0f}")
print(f"  predicted TOTALDOL: ${focus_meta['pred_gbr']:,.0f}")


## Conclusion

The Home Energy Copilot pipeline runs end-to-end on RECS 2020 across two modeling layers:

- A **multinomial classifier** sorts households into efficient / average / inefficient classes within their climate peer group, achieving weighted accuracy ≈ 60% (well above the 33% random baseline).
- A **gradient-boosted regressor** estimates annual energy spend with weighted RMSE ≈ $745 and R² ≈ 0.51, marginally outperforming Elastic Net on most climate groups.

**The two views are complementary, not redundant.** Classification answers *"how does this home compare to its climate peers in cost per square foot?"*; regression answers *"what absolute dollar amount should this household expect?"*. The cross-cutting analysis confirmed these are different signals — a household can be flagged as inefficient (small, drafty) while still having a modest total bill, and vice versa. That's by design and is honest about what each layer is measuring.

The system is not a replacement for an in-person energy audit, but it serves as a teaching example of how to stitch classification and regression on a real, weighted survey dataset where the target-engineering choices and the metric-weighting choices both matter for the conclusions.